# Step 0 · [3] Semantic filter

The "to review" families matched weak rules (B, C) or head-mounted / head-up display codes only. Here we read their abstracts with a language model and score how close each one is to our definition of smart glasses, as in Tutorial 0, Section 6.

1. **Embeddings.** Every abstract (title + abstract) becomes a vector with **Patent SBERT-adapt-ub** (`graziasveva93/sbert-adapt-ub`, Ascione & Sterzi 2024). We embed *all* candidate families, sure ones included: the vectors are reused later (subfields in Step 2, text novelty in Step 4).
2. **Scores**, all from `config/sector.yaml`:
   - `sim_definition`: cosine similarity with the `description` of smart glasses;
   - `sim_negative_max`: highest similarity with one of the **negative descriptions** (VR headset, vehicle HUD, contact lens, 3D viewing glasses, eyewear retail, conventional eyewear), and which one (`closest_negative`);
   - `margin = sim_definition − sim_negative_max`: positive when the abstract is closer to smart glasses than to any typical false positive;
   - `bm25`: lexical baseline, as in the tutorial.
3. **Check** on the sure families (rule A): they should score high.
4. **Labelling file.** A blind sample of to-review families across all margin levels, plus some sure families, to label by hand. The threshold is chosen from these labels in the next notebook (precision / recall / F1), not by eye.

**Runtime.** About 70k abstracts on 4 CPU cores: roughly 30–60 minutes. Embeddings are saved in chunks: if the kernel stops, run again and it continues where it stopped.

**Before running:** `03_texts` with the same `ENV`.

In [ ]:
%pip install -q pyyaml sentence-transformers rank-bm25

In [ ]:
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from src.sector import load_config

ENV = "TEST"                                    # same environment as 03_texts
OUTPUT_DIR = Path("outputs/step0")
CHUNK = 2000                                    # abstracts per saved chunk of embeddings
PER_BAND = 20                                   # to-review families to label per margin decile (10 deciles)
IN_SECTOR_SAMPLE = 40                           # sure families to label (precision of rule A)
SEED = 42
pd.set_option("display.max_colwidth", 110)

cfg = load_config()
sem = cfg["semantic"]


def read_table(stem):
    """Read outputs/step0/<stem>_<ENV> as parquet, or as CSV if parquet was not available."""
    path = OUTPUT_DIR / f"{stem}_{ENV}.parquet"
    return pd.read_parquet(path) if path.exists() else pd.read_csv(path.with_suffix(".csv"))


texts = read_table("texts")
docs = (texts[texts["has_abstract"]]
        .sort_values("docdb_family_id")
        .reset_index(drop=True))
docs["tier"] = np.where(docs["in_sector"], "in sector", "to review")
docs["doc"] = docs["appln_title"].fillna("") + ". " + docs["appln_abstract"].fillna("")
print(f"env {ENV} | {len(docs):,} families with an English abstract:")
print(docs["tier"].value_counts().to_string())

## 1. Embeddings (saved in chunks)

In [ ]:
EMB_DIR = OUTPUT_DIR / f"embeddings_{ENV}"
EMB_DIR.mkdir(parents=True, exist_ok=True)

# The chunks are only valid for this exact list of families: start again if 03_texts changed
ids = docs["docdb_family_id"].to_numpy()
ids_path = EMB_DIR / "family_ids.npy"
if ids_path.exists() and not np.array_equal(np.load(ids_path, allow_pickle=False), ids):
    for f in EMB_DIR.glob("chunk_*.npy"):
        f.unlink()
    print("the list of families changed: old embeddings removed")
np.save(ids_path, ids)

model = SentenceTransformer(sem["model"])
model.max_seq_length = sem["max_seq_length"]

starts = range(0, len(docs), CHUNK)
t0 = time.time()
for i, start in enumerate(starts, 1):
    path = EMB_DIR / f"chunk_{start:06d}.npy"
    if path.exists():
        continue
    emb = model.encode(docs["doc"].iloc[start:start + CHUNK].tolist(), batch_size=32,
                       normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    np.save(path, emb.astype(np.float32))
    print(f"  chunk {i}/{len(starts)} done, {(time.time() - t0) / 60:.1f} min")

E = np.vstack([np.load(EMB_DIR / f"chunk_{start:06d}.npy") for start in starts])
print("embedding matrix:", E.shape)

## 2. Scores

With normalised vectors the dot product is the cosine similarity.

In [ ]:
negative_names = list(sem["negatives"])
definition = model.encode(cfg["description"], normalize_embeddings=True, convert_to_numpy=True)
negatives = model.encode([sem["negatives"][n] for n in negative_names], normalize_embeddings=True, convert_to_numpy=True)

docs["sim_definition"] = E @ definition
negative_sims = E @ negatives.T
docs["sim_negative_max"] = negative_sims.max(axis=1)
docs["closest_negative"] = np.array(negative_names)[negative_sims.argmax(axis=1)]
docs["margin"] = docs["sim_definition"] - docs["sim_negative_max"]

TOKEN = re.compile(r"[a-z0-9]+")


def tokenize(text):
    return TOKEN.findall(text.lower())


bm25 = BM25Okapi([tokenize(d) for d in docs["doc"]])
docs["bm25"] = bm25.get_scores(tokenize(cfg["description"]))

score_cols = ["sim_definition", "margin", "bm25"]
print(docs.groupby("tier")[score_cols].describe(percentiles=[0.1, 0.5, 0.9]).T.round(3).to_string())

## 3. Check on the sure families

If the model and the definition work, the sure families (rule A) sit mostly on the right of the to-review ones, and most have a positive margin.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, score_cols):
    for tier, grp in docs.groupby("tier"):
        ax.hist(grp[col], bins=60, density=True, alpha=0.5, label=tier)
    ax.set(title=col, xlabel="score", ylabel="density")
    ax.legend()
axes[1].axvline(0, color="grey", lw=0.8)
plt.tight_layout()
plt.show()

share_positive = docs.groupby("tier")["margin"].apply(lambda m: (m > 0).mean()).round(3)
print("share with margin > 0:")
print(share_positive.to_string())
print()
print("closest negative description, to-review families with margin <= 0:")
print(docs[(docs["tier"] == "to review") & (docs["margin"] <= 0)]["closest_negative"].value_counts().to_string())

## 4. Eyeballing the to-review families

Highest margin, lowest margin, and around zero.

In [ ]:
review = docs[docs["tier"] == "to review"].sort_values("margin", ascending=False)
cols = ["margin", "closest_negative", "appln_title"]
mid = review.iloc[(review["margin"].abs()).argsort()[:10]] if len(review) else review

for label, part in (("highest margin", review.head(10)), ("around zero", mid), ("lowest margin", review.tail(10))):
    print(f"--- {label}")
    print(part[cols].to_string(index=False))

## 5. Labelling file

A **blind** sample: `PER_BAND` to-review families from each tenth of the margin distribution (so that every score level is represented), plus `IN_SECTOR_SAMPLE` random sure families (to measure the precision of rule A). The order is shuffled and the scores are **not** in the file, so the labeller is not influenced.

**How to label** (column `label`):

| value | when |
|---|---|
| `1` | smart glasses, as defined in `SECTOR_DEFINITION_PLAN.md` (4.1): eyewear with electronics (display, camera, sensors, audio, hearing) |
| `0` | not smart glasses: VR headset, vehicle/aircraft HUD, contact lens, 3D viewing glasses, ordinary spectacles, retail software, unrelated |
| empty | not sure: write why in `note` |

Borderline rules agreed so far: mixed-reality headsets with see-through optics and clip-on modules for glasses count as `1`; software and methods whose device is clearly smart glasses count as `1`.

Fill `labeler` with your initials. Ideally two people label the same file independently (we can then measure agreement).

**Files:**
- `outputs/step0/labels_to_fill_<ENV>.csv`: download it, label it (Excel or Google Sheets), and upload it back to TIP as `project/data/labels/labels_round1.csv` (keep the columns);
- `outputs/step0/labels_key_<ENV>.csv`: the scores of each sampled family, used by the next notebook. Do not open it before labelling.

In [ ]:
rng = np.random.default_rng(SEED)
review = docs[docs["tier"] == "to review"].copy()
review["band"] = pd.qcut(review["margin"], 10, labels=False, duplicates="drop") if len(review) >= 10 else 0
sample_review = review.groupby("band", group_keys=False).apply(
    lambda g: g.sample(min(len(g), PER_BAND), random_state=SEED))
sure = docs[docs["tier"] == "in sector"]
sample_sure = sure.sample(min(len(sure), IN_SECTOR_SAMPLE), random_state=SEED).assign(band=-1)

sample = pd.concat([sample_review, sample_sure]).sample(frac=1, random_state=SEED).reset_index(drop=True)
sample.insert(0, "label_id", np.arange(1, len(sample) + 1))

to_fill = sample[["label_id", "docdb_family_id", "appln_auth", "appln_title", "appln_abstract"]].assign(label="", labeler="", note="")
key = sample[["label_id", "docdb_family_id", "tier", "band", "sim_definition", "sim_negative_max", "closest_negative", "margin", "bm25"]]

to_fill.to_csv(OUTPUT_DIR / f"labels_to_fill_{ENV}.csv", index=False, encoding="utf-8-sig")   # utf-8-sig: opens cleanly in Excel
key.to_csv(OUTPUT_DIR / f"labels_key_{ENV}.csv", index=False)
print(f"{len(to_fill)} families to label: {len(sample_review)} to review + {len(sample_sure)} sure")
print(f"saved {OUTPUT_DIR / f'labels_to_fill_{ENV}.csv'} and the key")

## 6. Save the scores

One row per family with an English abstract. Stays on TIP (`outputs/` is git-ignored).

In [ ]:
out = docs[["docdb_family_id", "year", "tier", "sim_definition", "sim_negative_max", "closest_negative", "margin", "bm25"]]
path = OUTPUT_DIR / f"scores_{ENV}.parquet"
try:
    out.to_parquet(path, index=False)
except ImportError:                       # no pyarrow: fall back to CSV
    path = path.with_suffix(".csv")
    out.to_csv(path, index=False)
print(f"saved {len(out):,} scores to {path}")